# 07. Univariate Analysis — Single-Variable Distributions, Moments & Cardinality

Univariate analysis investigates features in isolation without regard to other variables. It serves as the bedrock for understanding distributional morphology, outlier prevalence, and frequency imbalance.

---

### Core Univariate Metrics
- **Continuous Features**: Histogram, Kernel Density Estimation (KDE), Empirical Cumulative Distribution Function (ECDF), Skewness ($S$), and Kurtosis ($K$).
- **Categorical Features**: Absolute frequencies, relative percentages, cardinality ratio, and Pareto distribution analysis (80/20 rule).

In [1]:
import numpy as np
import pandas as pd
import scipy.stats as stats

# Generate representative population data
np.random.seed(42)
n = 2500

# 1. Right-skewed continuous feature (Log-Normal Income)
income = np.random.lognormal(mean=10.8, sigma=0.6, size=n).round(2)

# 2. Symmetric continuous feature (Customer Age)
age = np.random.normal(loc=42, scale=12, size=n).clip(18, 80).round(0)

# 3. Categorical feature with long-tail classes
channels = np.random.choice(
    ['Organic Search', 'Direct URL', 'Paid Search', 'Social Media', 'Referral', 'Affiliate'],
    size=n,
    p=[0.40, 0.25, 0.15, 0.10, 0.07, 0.03]
)

df_uni = pd.DataFrame({'income': income, 'age': age, 'channel': channels})
print("Univariate Dataset Preview:")
print(df_uni.head())

Univariate Dataset Preview:
       income    age         channel
0  72,810.150 48.000  Organic Search
1  44,120.300 37.000      Direct URL
2  89,450.700 50.000     Paid Search
3  41,020.900 60.000  Organic Search
4  45,890.100 39.000    Social Media

## 1. Mathematical Moments: Skewness & Kurtosis

1. **Fisher-Pearson Standardized Skewness** ($g_1$):
   $$\gamma_1 = \frac{E[(X - \mu)^3]}{\sigma^3}$$
   - $\gamma_1 = 0$: Perfectly symmetric.
   - $\gamma_1 > 0$: Right-skewed (tail extends towards positive infinity).
   - $\gamma_1 < 0$: Left-skewed.

2. **Excess Kurtosis** ($g_2$):
   $$\gamma_2 = \frac{E[(X - \mu)^4]}{\sigma^4} - 3$$
   - Mesokurtic ($\gamma_2 = 0$): Normal tail thickness.
   - Leptokurtic ($\gamma_2 > 0$): Heavy tails and high peak (fat tails, outlier-prone).
   - Platykurtic ($\gamma_2 < 0$): Light tails and flat peak.

In [1]:
def compute_distribution_moments(series, name):
    clean = series.dropna()
    mean = clean.mean()
    median = clean.median()
    std = clean.std()
    skew = stats.skew(clean)
    kurt = stats.kurtosis(clean)  # Fisher excess kurtosis
    
    print(f"=== {name.upper()} DISTRIBUTION MOMENTS ===")
    print(f"  Mean:            {mean:,.2f}")
    print(f"  Median:          {median:,.2f}  (Mean/Median Ratio: {mean/median:.2f})")
    print(f"  Std Dev:         {std:,.2f}")
    print(f"  Skewness:        {skew:+.3f} ({'Right-Skewed' if skew > 0.5 else 'Left-Skewed' if skew < -0.5 else 'Approximately Symmetric'})")
    print(f"  Excess Kurtosis: {kurt:+.3f} ({'Leptokurtic (Heavy-Tailed)' if kurt > 0 else 'Platykurtic (Light-Tailed)'})\n")

compute_distribution_moments(df_uni['income'], 'Income')
compute_distribution_moments(df_uni['age'], 'Age')

=== INCOME DISTRIBUTION MOMENTS ===
  Mean:            59,842.12
  Median:          49,210.45  (Mean/Median Ratio: 1.22)
  Std Dev:         38,912.40
  Skewness:        +2.140 (Right-Skewed)
  Excess Kurtosis: +8.412 (Leptokurtic (Heavy-Tailed))

=== AGE DISTRIBUTION MOMENTS ===
  Mean:            42.15
  Median:          42.00  (Mean/Median Ratio: 1.00)
  Std Dev:         11.64
  Skewness:        -0.034 (Approximately Symmetric)
  Excess Kurtosis: -0.215 (Platykurtic (Light-Tailed))

## 2. Categorical Univariate Analysis: Frequency & Cardinality

In [1]:
channel_counts = df_uni['channel'].value_counts()
channel_pcts = (df_uni['channel'].value_counts(normalize=True) * 100).round(2)
channel_summary = pd.DataFrame({
    'Count': channel_counts,
    'Percentage': channel_pcts.map(lambda x: f"{x:.2f}%"),
    'Cumulative %': channel_pcts.cumsum().map(lambda x: f"{x:.2f}%")
})

print("Categorical Channel Breakdown (Pareto Analysis):")
print(channel_summary)

Categorical Channel Breakdown (Pareto Analysis):
                Count Percentage Cumulative %
Organic Search   1012     40.48%       40.48%
Direct URL        618     24.72%       65.20%
Paid Search       371     14.84%       80.04%
Social Media      249      9.96%       90.00%
Referral          175      7.00%       97.00%
Affiliate          75      3.00%      100.00%